Must run with DynamicGating environment (importing Arjun's fork of the SDK)

In [ ]:
import sys, os
# Imports of vbn_utils, ccf_utils, decoding_utils, analysis_utils,
# notebook_utils, vbn_4day_utils resolve via vbn_code/utilities/.
_UTILS_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'utilities'))
if _UTILS_DIR not in sys.path:
    sys.path.insert(0, _UTILS_DIR)

import pandas as pd
import numpy as np
import os
import glob
from matplotlib import pyplot as plt
from allensdk.brain_observatory.behavior.behavior_project_cache.\
    behavior_neuropixels_project_cache \
    import VisualBehaviorNeuropixelsProjectCache
from allensdk.brain_observatory.ecephys.behavior_ecephys_session import BehaviorEcephysSession
from analysis_utils import exponential_convolve
from scipy.stats import kstest, ranksums
from statsmodels.stats.multitest import multipletests
import vbn_utils
import vbn_4day_utils
import decoding_utils as du
%matplotlib inline
from notebook_utils import fwhm, fraction_above_half_max, get_peak_time


In [ ]:
high_res = True
if high_res:
    plt.rcParams['figure.dpi'] = 300
    plt.rcParams['savefig.dpi'] = 300
    plt.rcParams['font.size'] = 12

## Data loading

In [ ]:
probe_info = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/VBN_four_day_experiment_nwbs/probes.csv")
session_info = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/VBN_four_day_experiment_nwbs/sessions.csv")
all_units = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/VBN_four_day_experiment_nwbs/units.csv")

## Annotate sessions with useful metadata

In [ ]:
for session, sess_info in session_info.iterrows():

    dirname = sess_info['session_path']
    exp_id = sess_info['exp_id']
    nwb_path = glob.glob(os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/VBN_four_day_experiment_nwbs", 
                                      str(exp_id),
                                      '*.nwb'))[0]
    session_info.loc[session, 'nwb_path'] = nwb_path

    area_calls = glob.glob(os.path.join(dirname, '*areaClassifications.csv'))[0]
    area_calls = pd.read_csv(area_calls)

    for ip, proberow in area_calls.iterrows():
        probe = proberow['Probe']

        probe_info_index = probe_info[(probe_info['probe_name']==probe) & (probe_info['session_id']==sess_info['exp_id'])].index
        if len(probe_info_index) == 0:
            continue

        probe_info_index = probe_info_index[0]
        probe_area = proberow['Area']

        probe_info.loc[probe_info_index, 'area'] = probe_area

## Unit quality filtering

In [ ]:
good_session_info = session_info[~session_info['exp_id'].isin([1381868515,1382052219,1382241611,1382465116])]

## Overview PSTH by image set

In [ ]:
in_ctx = all_units['in_cortex']
quality = all_units['quality'] #this label includes standard quality metrics for this dataset
no_seizure = ~all_units['session_id'].isin([1381868515,1382052219,1382241611,1382465116])
in_vis_area = all_units['area'].isin(['VISp', 'VISl', 'VISrl', 'VISam', 'VISpm', 'VISal'])
#in_vis_area = all_units['area']=='VISp'
rs = all_units['waveform_duration'] > 0.4
unit_ids = all_units[quality & in_ctx & no_seizure & in_vis_area & rs]['unit_id'].values

In [ ]:
stim_filter = ['active', 'is_change', '~image_name.isin(["im083_r", "im111_r"])']
gpsth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['image_set']=='G']['exp_id'].values, unit_ids, stim_filter, win_before=1, win_after=1, bin_size=0.005)
stim_filter = ['active', 'is_change', '~image_name.isin(["im083_r", "im111_r"])']
h1psth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_2']['exp_id'].values, unit_ids, stim_filter, win_before=1, win_after=1, bin_size=0.005)
h2psth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_3']['exp_id'].values, unit_ids, stim_filter, win_before=1, win_after=1, bin_size=0.005)
stim_filter = ['active', 'is_change', '~image_name.isin(["im024_r", "im034_r"])']
kpsth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_4']['exp_id'].values, unit_ids, stim_filter, win_before=1, win_after=1, bin_size=0.005)


In [ ]:
plt.rcParams['font.size'] = 16
fig, ax = plt.subplots()
for ipsth, (psth, color) in enumerate(zip([gpsth, h1psth, h2psth, kpsth], ('b', 'r', 'purple', 'coral'))):
    goodpsth = [g for g in psth if len(g) > 0]
    grand_psth = np.concatenate(goodpsth, axis=0)
    
    base_end = int(1/0.005) #win_before / bin_size
    base_start = base_end - int(0.1/0.005) #0.1s baseline
    base_sub = grand_psth - grand_psth[:,base_start:base_end].mean(axis=1)[:,np.newaxis]

    mean = base_sub.mean(axis=0)
    sem = base_sub.std(axis=0)/np.sqrt(base_sub.shape[0])

    ax.plot(bins-1, mean, label=f'{ipsth}', color=color)
    ax.fill_between(bins-1, mean-sem, mean+sem, alpha=0.5, color=color)

ax.legend()
ax.set_xlim(0, 0.3)
vbn_utils.formatFigure(fig, ax, xLabel='Firing rate (Hz)', yLabel='Time from stimulus onset (s)')


## Nonchange flash PSTHs

In [ ]:
stim_filter = [ '~is_change',
                '~omitted', 
                '~previous_omitted',
                'flashes_since_change>5',
                'flashes_since_last_lick>1',]
                # 'image_name.isin(["im024_r", "im034_r"])']
bin_size = 0.005
win_before = 1
win_after = 1
gpsth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['image_set']=='G']['exp_id'].values, unit_ids, stim_filter + ['~image_name.isin(["im083_r", "im111_r"])'], cut=False, win_before=win_before, win_after=win_after, bin_size=bin_size)
h1psth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_2']['exp_id'].values, unit_ids, stim_filter + ['~image_name.isin(["im083_r", "im111_r"])'], cut=False, win_before=win_before, win_after=win_after, bin_size=bin_size)
h2psth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_3']['exp_id'].values, unit_ids, stim_filter + ['~image_name.isin(["im083_r", "im111_r"])'], cut=False, win_before=win_before, win_after=win_after, bin_size=bin_size)
kpsth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_4']['exp_id'].values, unit_ids, stim_filter + ['~image_name.isin(["im024_r", "im034_r"])'], cut=False, win_before=win_before, win_after=win_after, bin_size=bin_size)


In [ ]:
time_to_peak = np.argmax(base_sub[:,base_start:base_start+int(0.3/bin_size)], axis=1)*bin_size
plt.figure()
plt.hist(time_to_peak, bins=50)

## Response properties: peak time, amplitude, FWHM

In [ ]:
win_before = 1
win_after = 1
bin_size = 0.005
bins = np.arange(-win_before, win_after, bin_size)
base_end = int(win_before/bin_size)
stim_end = int((win_before + 0.25)/bin_size)
resp_slice = slice(base_end + int(0.010/bin_size), stim_end + int(0.050/bin_size))

fig, ax = plt.subplots()
fig_peak_time, ax_peak_time = plt.subplots(constrained_layout=True)
fig_peak_amp, ax_peak_amp = plt.subplots()
fig_fwhm, ax_fwhm = plt.subplots()
peak_times = []
peak_amps = []
fwhms = []
for ipsth, (psth, color, imlabel) in enumerate(zip([gpsth, h1psth, h2psth, kpsth], ('b', 'r', 'purple', '#FBAF41'), ('G', 'H1', 'H2', 'K'))):
    goodpsth = [g for g in psth if len(g) > 0]
    grand_psth = np.concatenate(goodpsth, axis=0)
    base_end = int(win_before/bin_size)
    base_start = base_end - int(0.1/bin_size)  # 0.1s baseline
    base_sub = grand_psth - grand_psth[:,base_start:base_end].mean(axis=1)[:,np.newaxis]

    mean = base_sub.mean(axis=0)
    sem = base_sub.std(axis=0)/np.sqrt(base_sub.shape[0])
    ax.plot(bins, mean, label=f'{imlabel}: n={base_sub.shape[0]}', color=color)
    ax.fill_between(bins, mean-sem, mean+sem, alpha=0.5, color=color, lw=0)

    time_to_peak = np.array([get_peak_time(trace, slice(resp_slice.start, int(resp_slice.start + 0.150/bin_size)), bin_size) for trace in base_sub]) - win_before
    ax_peak_time.boxplot(time_to_peak, positions=[ipsth], widths=0.5, showfliers=False, whis=[10,90], notch=True, patch_artist=True,
        boxprops=dict(facecolor=color, color=color), whiskerprops=dict(color=color),
        capprops=dict(color=color), medianprops=dict(color='w', linewidth=2))
    ax_peak_time.set_ylabel('peak time (s)')
    ax_peak_time.set_xticks(np.arange(4))
    ax_peak_time.set_xticklabels(['G', 'H1', 'H2', 'K'])
    peak_times.append(time_to_peak)

    peak_amp = base_sub[:,base_end:base_end+int(0.1/bin_size)].mean(axis=1)
    peak_amp = peak_amp[~np.isnan(peak_amp)]
    ax_peak_amp.boxplot(peak_amp, positions=[ipsth], widths=0.5, showfliers=False, whis=[10,90], notch=True, patch_artist=True,
        boxprops=dict(facecolor=color, color=color), whiskerprops=dict(color=color),
        capprops=dict(color=color), medianprops=dict(color='w', linewidth=2))
    ax_peak_amp.set_ylabel('peak amplitude (Hz)')
    ax_peak_amp.set_xticks(np.arange(4))
    ax_peak_amp.set_xticklabels(['G', 'H1', 'H2', 'K'])
    peak_amps.append(peak_amp)

    full_width_half_max = np.array([fraction_above_half_max(bin_size, trace) for trace in base_sub[:,base_end:base_end+int(0.3/bin_size)]])
    ax_fwhm.boxplot(full_width_half_max[~np.isnan(full_width_half_max)], positions=[ipsth], widths=0.5, showfliers=False, whis=[10,90], notch=True, patch_artist=True,
        boxprops=dict(facecolor=color, color=color), whiskerprops=dict(color=color),
        capprops=dict(color=color), medianprops=dict(color='w', linewidth=2))
    ax_fwhm.set_ylabel('fwhm (s)')
    fwhms.append(full_width_half_max[~np.isnan(full_width_half_max)])
    ax_fwhm.set_xticks(np.arange(4))
    ax_fwhm.set_xticklabels(['G', 'H1', 'H2', 'K'])

ax.set_xlim(0, 0.3)
ax.legend(frameon=False)
ax.set_xlabel('Time from stimulus onset (s)')
ax.set_ylabel('Firing rate (Hz)')
vbn_utils.formatFigure(fig, ax)
vbn_utils.formatFigure(fig_peak_time, ax_peak_time)
vbn_utils.formatFigure(fig_peak_amp, ax_peak_amp)
vbn_utils.formatFigure(fig_fwhm, ax_fwhm)
plt.tight_layout()

In [ ]:
from scipy.stats import kstest, ranksums
from statsmodels.stats.multitest import multipletests

In [ ]:
for vals, ax in zip([peak_times, peak_amps, fwhms], [ax_peak_time, ax_peak_amp, ax_fwhm]):
    pvalues = []
    comparisons = []
    for ip1, p1 in enumerate(vals):
        for ip2, p2 in enumerate(vals[ip1+1:]):
            print(f'{ip1} vs {ip2+ip1+1}: {ranksums(p1, p2)}')
            pvalues.append(ranksums(p1, p2).pvalue)
            comparisons.append((ip1, ip2+ip1+1))

    corrected_pvalues = multipletests(pvalues, method='fdr_bh')[1]
    ax_max = ax.get_ylim()[1]
    y_increment = ax_max/10
    for i, (start, end) in enumerate(comparisons):
        if corrected_pvalues[i]<0.05:
            y = ax_max + y_increment*i
            ax.plot([start, start, end, end], [y, y + y_increment/2, y + y_increment/2, y], lw=1.5, c='k',)
            ax.text((start + end) * .5, y+ y_increment/2, f"p = {corrected_pvalues[i]:.1e}", ha='center', va='bottom', color='k', fontdict={'size': 14})
    plt.tight_layout()

## Active vs. passive PSTHs by image set

In [ ]:
stim_filter = [ '~is_change',
                '~omitted', 
                '~previous_omitted',
                '~grace_period_after_hit',
                'flashes_since_last_lick>1']
gpsth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['image_set']=='G']['exp_id'].values, unit_ids, stim_filter + ['~image_name.isin(["im083_r", "im111_r"])'], cut=True, win_before=1, win_after=1, bin_size=0.01)
h1psth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_2']['exp_id'].values, unit_ids, stim_filter + ['~image_name.isin(["im083_r", "im111_r"])'], cut=True, win_before=1, win_after=1, bin_size=0.01)
h2psth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_3']['exp_id'].values, unit_ids, stim_filter + ['~image_name.isin(["im083_r", "im111_r"])'], cut=True, win_before=1, win_after=1, bin_size=0.01)
kpsth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_4']['exp_id'].values, unit_ids, stim_filter + ['~image_name.isin(["im024_r", "im034_r"])'], cut=True, win_before=1, win_after=1, bin_size=0.01)


In [ ]:
stim_filter = [ '~is_change', 
                '~omitted', 
                '~previous_omitted',
                '~grace_period_after_hit',
                'flashes_since_last_lick>1']
shared1_gpsth, bins =  vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_1']['exp_id'].values, unit_ids, stim_filter + ['image_name.isin(["im083_r", "im111_r"])'], cut=True, win_before=1, win_after=1, bin_size=0.01)
shared1_h1psth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_2']['exp_id'].values, unit_ids, stim_filter + ['image_name.isin(["im083_r", "im111_r"])'], cut=True, win_before=1, win_after=1, bin_size=0.01)
shared1_h2psth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_3']['exp_id'].values, unit_ids, stim_filter + ['image_name.isin(["im083_r", "im111_r"])'], cut=True, win_before=1, win_after=1, bin_size=0.01)

shared2_h1psth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_2']['exp_id'].values, unit_ids, stim_filter + ['image_name.isin(["im024_r", "im034_r"])'], cut=True, win_before=1, win_after=1, bin_size=0.01)
shared2_h2psth, bins = vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_3']['exp_id'].values, unit_ids, stim_filter + ['image_name.isin(["im024_r", "im034_r"])'], cut=True, win_before=1, win_after=1, bin_size=0.01)
shared2_kpsth, bins =  vbn_4day_utils.unit_averaged_psth_from_nwb(good_session_info[good_session_info['stage']=='EPHYS_day_4']['exp_id'].values, unit_ids, stim_filter + ['image_name.isin(["im024_r", "im034_r"])'], cut=True, win_before=1, win_after=1, bin_size=0.01)

## Aggregate responses by image set and condition

In [ ]:
aggregated = {image_set: {cond: [] for cond in ['active','passive']} for image_set in ['G', 'H1', 'H2', 'K']}
for psth, imset in zip([gpsth, h1psth, h2psth, kpsth], ['G', 'H1', 'H2', 'K']):
    for condind, cond in zip([0,1], ['active', 'passive']):
        psth_cond = np.array(psth)[:, condind]
        stacked = []
        for p in psth_cond:
            stacked.append(np.stack(p))
        
        aggregated[imset][cond]=np.concatenate(stacked, axis=1)


In [ ]:
fig, ax = plt.subplots(1, 4)
for i, imset in enumerate(['G', 'H1', 'H2', 'K']):
    for cond, color in zip(['active', 'passive'], ['b', 'gray']):
        for cut in range(5):
            data = aggregated[imset][cond][cut].mean(axis=0)
            plot_data = data-data[:20].mean()
            ax[i].plot(plot_data, alpha=1/(cut+1), color=color, label=f'{cond} cut {cut}')
    ax[i].set_title(imset)
    ax[i].set_ylim([-2, 12])
    ax[i].set_xlim([100, 130])
    #ax[i].legend()

In [ ]:
aggregated = {image_set: {cond: {imsubset:[] for imsubset in ['shared1', 'shared2', 'nonshared']} for cond in ['active','passive']} for image_set in ['G', 'H1', 'H2', 'K']}
for psth, imset, imsubset in zip([gpsth, h1psth, h2psth, kpsth, shared1_gpsth, shared1_h1psth, shared1_h2psth, shared2_h1psth, shared2_h2psth, shared2_kpsth], 
                        ['G', 'H1', 'H2', 'K', 'G', 'H1', 'H2', 'H1', 'H2', 'K'],
                        ['nonshared', 'nonshared', 'nonshared', 'nonshared', 'shared1', 'shared1', 'shared1', 'shared2', 'shared2', 'shared2']):
    for condind, cond in zip([0,1], ['active', 'passive']):
        psth_cond = np.array(psth)[:, condind]
        stacked = []
        for p in psth_cond:
            stacked.append(np.stack(p))
        
        aggregated[imset][cond][imsubset]=np.concatenate(stacked, axis=1)


In [ ]:
fig, ax = plt.subplots(1, 4)
fig.suptitle('Nonshared')
for i, imset in enumerate(['G', 'H1', 'H2', 'K']):
    for cond, color in zip(['active', 'passive'], ['b', 'gray']):
        for cut in range(5):
            data = aggregated[imset][cond]['nonshared'][cut].mean(axis=0)
            plot_data = data-data[:20].mean()
            ax[i].plot(plot_data, alpha=1/(cut+1), color=color, label=f'{cond} cut {cut}')
    ax[i].set_title(imset)
    ax[i].set_ylim([-2, 15])
    ax[i].set_xlim([100, 130])
    #ax[i].legend()

fig, ax = plt.subplots(1, 4)
fig.suptitle('Shared')
for i, imset in enumerate(['G', 'H1', 'H2', 'K']):
    for cond, color in zip(['active', 'passive'], ['b', 'gray']):
        for imsubset, ls in zip(['shared1', 'shared2'], ['solid', 'dashed']):
            data = aggregated[imset][cond][imsubset]
            for cut in range(5):
                if len(data)>0:
                    plot_data = data[cut].mean(axis=0)
                    plot_data = plot_data-plot_data[:20].mean()
                    ax[i].plot(plot_data, alpha=1/(cut+1), color=color, ls=ls, label=f'{cond} cut {cut}')
    ax[i].set_title(imset)
    ax[i].set_ylim([-2, 15])
    ax[i].set_xlim([100, 130])

## Early vs. late session responses

In [ ]:
eps, bins = vbn_4day_utils.psth_from_nwb_cut(1375871900, unit_ids, ['~image_name.isin(["im083_r", "im111_r"])', 
                                                                    '~is_change', 
                                                                    '~omitted', 
                                                                    '~previous_omitted',
                                                                    '~grace_period_after_hit',
                                                                    'flashes_since_last_lick>1'])

In [ ]:
eps = np.array(eps)
fig, axes = plt.subplots(1,2)
for ep, ax in zip(eps, axes):
    for ic, cut in enumerate(ep):
        ax.plot(bins, cut.mean(axis=0), label=ic)

[ax.set_ylim(2, 18) for ax in axes]
[ax.legend() for ax in axes]